In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.stats import mode

### Fetch MNIST data

In [29]:
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
y = y.astype(int)
X = X / 255.0

### PCA to reduce dimensions

In [23]:
pca = PCA(n_components=0.95, random_state=42)
X_reduced = pca.fit_transform(X)

### Fit K-means++

In [24]:
kmeans = KMeans(n_clusters=10, init='k-means++', n_init=10, random_state=42)
cluster_assignments = kmeans.fit_predict(X_reduced)

### Predict labels

In [25]:
predicted_labels = np.zeros_like(cluster_assignments)
for i in range(10):
    mask = (cluster_assignments == i)
    if np.any(mask):
        dominant_label = mode(y[mask], keepdims=True).mode[0]
        predicted_labels[mask] = dominant_label

### Calculate accuracy

In [26]:
accuracy = accuracy_score(y, predicted_labels)
print(f"\nK-means results")
print(f"Mapped K-means Clustering Accuracy: {accuracy * 100:.2f}%")


K-means results
Mapped K-means Clustering Accuracy: 58.49%


### DBSCAN + UMAP dimensionality reduction

In [7]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score, silhouette_score
import umap


### UMAP

In [8]:
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
X_embedded = reducer.fit_transform(X)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### DBSCAN clustering

In [9]:
dbscan = DBSCAN(eps=0.5, min_samples=10)
cluster_assignments = dbscan.fit_predict(X_embedded)

### Accuracy

In [12]:
predicted_labels = np.zeros_like(cluster_assignments)
unique_clusters = set(cluster_assignments)

for cluster_num in unique_clusters:
    if cluster_num == -1:
        continue
    mask = (cluster_assignments == cluster_num)
    dominant_label = mode(y[mask], keepdims=True).mode[0]
    predicted_labels[mask] = dominant_label

predicted_labels[cluster_assignments == -1] = -1

total_accuracy = accuracy_score(y, predicted_labels)
print(f"Mapped DBSCAN Accuracy: {total_accuracy * 100:.2f}%")

Mapped DBSCAN Accuracy: 60.50%


### Aglomerative Hierarchical clustering

In [27]:
from sklearn.cluster import AgglomerativeClustering

np.random.seed(42)
indices = np.random.choice(len(X), 10000, replace=False)
X_sub, y_sub = X[indices] / 255.0, y[indices]

pca = PCA(n_components=0.95, random_state=42)
X_reduced = pca.fit_transform(X_sub)

agg_clustering = AgglomerativeClustering(n_clusters=10, linkage='ward')
cluster_assignments = agg_clustering.fit_predict(X_reduced)

predicted_labels = np.zeros_like(cluster_assignments)
for i in range(10):
    mask = (cluster_assignments == i)
    if np.any(mask):
        dominant_label = mode(y_sub[mask], keepdims=True).mode[0]
        predicted_labels[mask] = dominant_label

accuracy = accuracy_score(y_sub, predicted_labels)
print(f"\nHierarchical Clustering (Ward) Results")
print(f"Mapped Clustering Accuracy: {accuracy * 100:.2f}%")


Hierarchical Clustering (Ward) Results
Mapped Clustering Accuracy: 70.49%


Unlike K-means and Aglomerative Hierarchical clustering, DBSCAN doesn't know how many clusters there are (10), it uncovers them based on how many distinct density shapes it finds. Surprisingly, DBSCAN accuracy was better than the K-means (60.5% vs. 58.5%). **Aglomerative Hierarchical clustering** achived the best accuracy (70.5%) on lower number of samples (n=10000) to avoid O(n^2) memory issue.